# Phase 1: Step-by-Step Interactive Notebook
### Agentic AI Platform with Groq LLM & Multi-MCP Server (SQLite + Tavily Search)

This notebook demonstrates:
1. **Environment & UV Dependencies Setup**
2. **Creating the SQLite Business Database & Tables**
3. **Connecting via MCP (`MultiServerMCPClient` + `madnh/mcp-server-sqlite`)**
4. **Inspecting Tools exposed by the SQLite MCP Server**
5. **Executing MCP Tools with Groq LLM Agent (`ChatGroq`)**
6. **Chat History in Application Database**

## 1. Setup & Environment Variables
Make sure you have set your `GROQ_API_KEY`.

In [ ]:
import os
import sys
import asyncio
from dotenv import load_dotenv

# Fix for Jupyter kernel on Windows where sys.stderr is wrapped by OutStream without fileno()
if sys.platform == "win32" and hasattr(sys, "__stderr__") and sys.__stderr__ is not None:
    sys.stderr = sys.__stderr__
    sys.stdout = sys.__stdout__

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    print("⚠️ Please set GROQ_API_KEY in your .env file or environment variable.")
else:
    print("✅ GROQ_API_KEY detected!")

## 2. Initialize Sample Business Database (`data/business_data.db`)
Create tables: `customers`, `products`, `sales_orders` with sample business data.

In [ ]:
import sqlite3

DB_PATH = "./data/business_data.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Create products table
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER NOT NULL
)
""")

# Create customers table
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    region TEXT NOT NULL,
    join_date TEXT NOT NULL
)
""")

# Create sales_orders table
cursor.execute("""
CREATE TABLE IF NOT EXISTS sales_orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    total_amount REAL NOT NULL,
    order_date TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
""")

# Clear and insert sample records
cursor.execute("DELETE FROM sales_orders")
cursor.execute("DELETE FROM products")
cursor.execute("DELETE FROM customers")

cursor.executemany("""
INSERT INTO products (product_id, name, category, price, stock_quantity) VALUES (?, ?, ?, ?, ?)
""", [
    (1, "AI Developer Laptop Pro", "Electronics", 1899.99, 45),
    (2, "Cloud GPU Workstation", "Hardware", 4299.00, 12),
    (3, "4K Ultra Monitor 32inch", "Electronics", 499.50, 80),
    (4, "Mechanical Ergonomic Keyboard", "Accessories", 129.99, 150),
    (5, "Noise-Canceling AI Headset", "Accessories", 199.99, 95),
    (6, "Enterprise LLM Gateway Server", "Servers", 8900.00, 6)
])

cursor.executemany("""
INSERT INTO customers (customer_id, name, email, region, join_date) VALUES (?, ?, ?, ?, ?)
""", [
    (1, "Alice Smith", "alice@enterprise-ai.com", "North America", "2023-01-15"),
    (2, "Bob Jones", "bob@fintech-solutions.org", "Europe", "2023-04-20"),
    (3, "Carlos Santana", "carlos@globaltech.co", "Latin America", "2023-07-10"),
    (4, "Diana Prince", "diana@cloudscale.io", "Asia-Pacific", "2023-11-05"),
    (5, "Evan Wright", "evan@ai-research-lab.net", "North America", "2024-02-18")
])

cursor.executemany("""
INSERT INTO sales_orders (order_id, customer_id, product_id, quantity, total_amount, order_date) VALUES (?, ?, ?, ?, ?, ?)
""", [
    (101, 1, 1, 2, 3799.98, "2024-03-01"),
    (102, 1, 3, 2, 999.00, "2024-03-01"),
    (103, 2, 2, 1, 4299.00, "2024-03-15"),
    (104, 3, 4, 5, 649.95, "2024-04-02"),
    (105, 4, 5, 3, 599.97, "2024-04-10"),
    (106, 5, 6, 1, 8900.00, "2024-04-18"),
    (107, 2, 1, 1, 1899.99, "2024-05-05")
])

conn.commit()
conn.close()
print("✅ Sample database 'data/business_data.db' created successfully!")

## 3. Connect to `madnh/mcp-server-sqlite` MCP Server using `MultiServerMCPClient`
We launch the official `madnh/mcp-server-sqlite` server directly via `MultiServerMCPClient` using stdio transport:

In [ ]:
import os
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient

# Ensure Windows stdio handles use the underlying OS fileno
if sys.platform == "win32" and hasattr(sys, "__stderr__") and sys.__stderr__ is not None:
    sys.stderr = sys.__stderr__
    sys.stdout = sys.__stdout__

abs_db_path = os.path.abspath("./data/business_data.db")
sqlite_runner = os.path.abspath("backend/mcp/madnh_sqlite_runner.py")
tavily_server = os.path.abspath("backend/mcp/tavily_server.py")

env_vars = os.environ.copy()
if os.getenv("TAVILY_API_KEY"):
    env_vars["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

client = MultiServerMCPClient(
    {
        "sqlite": {
            "command": sys.executable,
            "args": [
                sqlite_runner,
                "--db-path",
                abs_db_path
            ],
            "transport": "stdio"
        },
        "tavily": {
            "command": sys.executable,
            "args": [
                tavily_server
            ],
            "env": env_vars,
            "transport": "stdio"
        }
    }
)

# Retrieve MCP tools
tools = await client.get_tools()

print(f"Loaded {len(tools)} tools from SQLite MCP Server:")
for tool in tools:
    print(f" - Name: {tool.name}")
    print(f"   Description: {tool.description}")

## 4. Bind MCP Tools to the Groq LLM Agent
We use `ChatGroq` (`llama-3.3-70b-versatile`) and bind the MCP tools.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

# Bind the MCP tools to the LLM
llm_with_tools = llm.bind_tools(tools)
print("✅ LLM configured with SQLite MCP tools!")

## 5. Ask Questions against your SQLite Database via MCP
Let's ask the agent to query the business tables.

In [ ]:
query = "What are the top 3 best-selling products by total revenue? List product name and total revenue."
messages = [
    SystemMessage(content="You are an expert SQL business analyst. Use the available SQLite MCP tools (list_tables, describe_table, read_query) to inspect schemas and query data accurately to answer the user's questions."),
    HumanMessage(content=query)
]

# Step 1: Model decides which MCP tool to call
ai_msg = await llm_with_tools.ainvoke(messages)
print("AI Response / Tool Calls:")
print(ai_msg.tool_calls)

# Step 2: Execute tool call if requested
tool_map = {tool.name: tool for tool in tools}
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    selected_tool = tool_map[tool_call["name"]]
    tool_output = await selected_tool.ainvoke(tool_call["args"])
    print(f"\n[Tool: {tool_call['name']}] Output:", tool_output)
    
    # Append tool output to conversation
    from langchain_core.messages import ToolMessage
    messages.append(ToolMessage(tool_call_id=tool_call["id"], content=str(tool_output)))

# Step 3: Get final synthesized response from LLM
final_response = await llm_with_tools.ainvoke(messages)
print("\n--- Final Answer ---")
print(final_response.content)

## 6. End-to-End Chat Loop with LangGraph or Direct Invocation

In [ ]:
async def ask_agent(user_question: str):
    conv = [
        SystemMessage(content="You are an enterprise AI assistant. Always inspect table schemas or execute SQL queries via MCP tools when asked about database records."),
        HumanMessage(content=user_question)
    ]
    
    # First call
    res = await llm_with_tools.ainvoke(conv)
    conv.append(res)
    
    # Handle tool calls
    while res.tool_calls:
        for tc in res.tool_calls:
            tool = tool_map[tc["name"]]
            out = await tool.ainvoke(tc["args"])
            conv.append(ToolMessage(tool_call_id=tc["id"], content=str(out)))
        res = await llm_with_tools.ainvoke(conv)
        conv.append(res)
        
    return res.content

# Test query
answer = await ask_agent("Show all customers from North America and how many orders each has placed.")
print("Result:\n", answer)